# Sources Matrix

This notebook loads the available source documents and builds the full matrix of scenario, model, and document combinations.

In [ ]:
from itertools import product
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root containing pyproject.toml")


repo_root = find_repo_root(Path.cwd())
documents_root = repo_root / "data" / "soruce" / "documents"
artifacts_root = repo_root / "artifacts" / "sources"
matrix_path = artifacts_root / "source-matrix.json"

models = [
    "deepseek/deepseek-chat-v3.1",
    "qwen/qwen3-235b-a22b",
    "openai/gpt-5-mini",
    "google/gemini-2.5-flash",
    "anthropic/claude-haiku-4.5",
    "google/gemini-2.5-pro",
]

ignored_document_names = {"README.md", "README.txt", "file_list.txt", "manifest.tsv"}
document_suffixes = {".md", ".txt"}
documents = sorted(
    path
    for path in documents_root.rglob("*")
    if path.is_file()
    and path.suffix.lower() in document_suffixes
    and path.name not in ignored_document_names
)
scenarios = ["control", "guided", "feedback"]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

matrix = pd.DataFrame(
    [
        {
            "scenario": scenario,
            "model": model,
            "document": document.name,
            "document_stem": document.stem,
            "document_path": document.relative_to(repo_root).as_posix(),
        }
        for scenario, model, document in product(scenarios, models, documents)
    ]
).sort_values(["scenario", "model", "document"]).reset_index(drop=True)

artifacts_root.mkdir(parents=True, exist_ok=True)
matrix.to_json(matrix_path, orient="records", indent=2, force_ascii=False)

print(f"Documents: {len(documents)}")
print(f"Scenarios: {len(scenarios)}")
print(f"Models: {len(models)}")
print(f"Total combinations: {len(matrix)}")
print(f"Saved matrix to: {matrix_path.relative_to(repo_root).as_posix()}")

display(matrix)
